In [2]:
from IPython.display import Image

- https://verl.readthedocs.io/en/latest/advance/agent_loop.html
    - https://blog.csdn.net/gitblog_01105/article/details/151440556
    - https://www.cnblogs.com/volcengine-developer/articles/19070102

> AgentLoop 只关注"怎么聊"/调用工具（逻辑），Worker 只关注"怎么调度"（并发），ServerManager 只关注"怎么生成"（算力），三者互不干扰。
- AgentLoopManager => AgentLoopWorker => Custom AgentLoop (重新 run 方法) => AsyncLLMServerManager
    - `agent_loop_manager.generate_sequences(prompts=batch)` 时，AgentLoopManager 会将任务分发给 AgentLoopWorker。
        - Manager 根据 Worker 的数量，将大的 batch 切分成多个小 chunk。
        - Manager 通过 Ray 的远程调用 worker.generate_sequences.remote(chunk) 将数据分发给分布在不同节点上的 Worker。
    - Worker 会检查数据 batch 中的 agent_name 字段（例如 "single_turn_agent" 或 "tool_agent"）。
    - Worker 使用 hydra.utils.instantiate 根据注册表（_agent_loop_registry）中对应的配置来实例化具体的 AgentLoop 类。
        - Worker 接收到数据后，遍历每一条样本。
        - Worker 读取样本中的 agent_name（例如 "tool_agent"）。
        - Worker 使用 hydra.utils.instantiate 动态实例化 对应的 AgentLoop 类（如 ToolAgentLoop）。
        - 关键点：在实例化时，Worker 会把自己持有的 server_manager、tokenizer 等基础设施对象注入给 AgentLoop。
        - Worker 调用 await agent_loop.run() 启动该样本的交互循环。
    - AgentLoop -> AsyncLLMServerManager (OpenAI compatible LLM server manager.)
        - AsyncLLMServerManager 是 LLM 服务的网关/负载均衡器。
            - AgentLoop 调用 server_manager.generate(prompt_ids, ...)。
            - ServerManager 内部维护了一组 rollout_replicas（即远程的 vLLM 引擎 Actor）。
            - ServerManager 根据负载均衡策略（如 Least Requests）选择一个可用的 vLLM 实例。
            - 通过 Ray 发送异步请求 server.generate.remote(...) 并等待 Token 返回。
    - Custom AgentLoop (重写 run 方法)
        - SingleTurnAgentLoop
        - ToolAgentLoop
        - ReactAgentLoop: LangGraph React Agent Loop.
- AsyncLLMServerManager 持有一组 server_handles（Ray Actor 句柄）。在 generate 方法中，它统一调用 server.generate.remote(...)。
- AsyncLLMServerManager 实例化流程
    - 步骤 1：注册 (Registry)
        - 在 verl/workers/rollout/replica.py 中，两个版本被注册到工厂类中：
            ```python
            RolloutReplicaRegistry.register("vllm", _load_vllm)
            RolloutReplicaRegistry.register("sglang", _load_sglang)
            ```
    - 步骤 2：选择 (Selection)
        - 在 `AgentLoopManager._initialize_llm_servers` 中：
        - 如果配置是 rollout.name=vllm，则获取 vLLMReplica 类。
        - 如果配置是 rollout.name=sglang，则获取 SGLangReplica 类。
    - 步骤 3：实例化 (Instantiation)
        ```python
        self.rollout_replicas = [
            self.rollout_replica_class(...) # 实例化具体的 Replica
            for replica_rank in range(num_replicas)
        ]
        ```
    - 步骤 4：注入 (Injection)
        - 实例化后的 server_handles 被传递给 AsyncLLMServerManager：
        ```python
        self.server_handles = [server._server_handle for server in self.rollout_replicas]
        # ...
        self.agent_loop_workers.append(
            # 将 server_handles 传给 Worker，Worker 再传给具体的 AgentLoop
            self.agent_loop_workers_class.options(...).remote(..., self.server_handles, ...)
        )
        ```

```mermaid
graph TD
    A[AgentLoopManager] -->|1. 切分 Batch & 分发| B(AgentLoopWorker)
    B -->|2. 实例化 & 调用 run| C{AgentLoop <br> e.g. ToolAgentLoop}
    C -->|3. 请求生成| D[AsyncLLMServerManager]
    D -->|4. RPC 调用| E[Remote vLLM Engine]
    C -->|5. 执行工具| F[Tools / Environment]
```

- 输入：DataProto (Batch Prompts) 进入 Manager。
- 分流：拆分后进入 Worker。
- 处理：Worker 启动 AgentLoop，AgentLoop 在循环中多次与 ServerManager 交互（Prompt -> Tokens -> Tool Result -> Prompt -> ...）。
- 输出：AgentLoop 返回包含完整轨迹（Prompt + Response + Tool Output）的 AgentLoopOutput 给 Worker。
- 聚合：Worker 将结果 Pad 成 Tensor，打包回传给 Manager。
- 最终结果：Manager 拼合所有 Worker 的结果，返回完整的 DataProto。

### Server-based的请求异步rollout

In [3]:
Image(url='https://github.com/eric-haibin-lin/verl-community/blob/main/docs/agent_loop_architecture.png?raw=true', width=700)

1. 任务发起与调度 (控制层)
- ① generate_sequences (PPOTrainer -> AgentLoopManager):
    - 流程始于 TaskRunner 中的 PPOTrainer。这是训练的主控制器。
    - 当需要新的训练数据时，PPOTrainer 向 AgentLoopManager 发出 generate_sequences 指令。这意味着：“我现在需要一批新的模型自我生成的对话数据（Rollout）来进行这一轮的参数更新。”
- ② wake_up (AgentLoopManager -> AsyncSglangServer/AsyncvLLMServer):
    - 在开始生成之前，管理器向推理服务器（AsyncServer）发送唤醒信号。
    - 在高效的 RLHF 框架中，显存是极其宝贵的。模型权重通常在“训练模式”（FSDP）和“推理模式”（vLLM/TP）之间复用（hybrid）。这个步骤可能涉及将 GPU 显存从训练状态切换或重新配置为推理状态。
        - wake_up 和 sleep 是 verl 在 Hybrid（混合）模式下实现高效资源利用的核心机制。
        - 这两个操作是为了解决 训练（Training） 和 推理（Rollout/Inference） 共享同一组 GPU 时的显存冲突和权重同步问题。
- ③ generate_sequences (AgentLoopManager -> AgentLoopWorker):
    - 管理器将生成任务分发给多个 AgentLoopWorker。这实现了数据并行，多个 Worker 同时处理不同的 Prompts（提示词）。
2. 交互与请求构建 (逻辑层)
- ④ prompts -> AgentLoop:
    - 在每个 Worker 内部，具体的 prompt（问题/上下文）被输入到 AgentLoop 中。
    - AgentLoop 负责维护强化学习的环境交互逻辑（例如：如果是多轮对话，它管理对话历史；如果是单轮，它准备输入）。
- ⑤ Call AsyncLLMServerManager:
    - 当 AgentLoop 需要模型根据 Prompt 生成回复时，它调用本地的 AsyncLLMServerManager。
- ⑥ generate(prompt_ids) (Manager -> AsyncServer):
    - Worker 层的管理器将具体的 Token ID 序列（prompt_ids）发送给中心化的 AsyncSglangServer 或 AsyncvLLMServer。
    - 这里使用了异步（Async）调用，允许 Worker 非阻塞地继续处理其他逻辑，直到推理结果返回。
3. 分布式推理执行 (计算层)
- ⑦ ipc/rpc (AsyncServer -> Model Runner):
    - 这是实际 GPU 计算发生的地方。推理服务器通过 IPC（进程间通信）或 RPC（远程过程调用）指挥底层的 Model Runner。
    - 关键架构细节：
        - FSDP Group (world_size=8): 表示总共有 8 个 GPU 参与（FSDP-0 到 FSDP-7），它们在训练阶段使用 FSDP（全分片数据并行）来切分大模型参数。
        - vLLM Group (tensor_parallel_size=4): 在推理阶段，这 8 个 GPU 被重新组织。如图所示，它们被分为两组，每组 4 个 GPU。这 4 个 GPU 使用 张量并行 (Tensor Parallelism) 协同工作来加速推理。
        - 这种设计（Training用FSDP，Inference用TP）是 verl 等框架的经典设计，旨在最大化训练时的显存效率和推理时的速度。
4. 资源释放
- ⑧ sleep (AgentLoopManager -> AsyncServer):
    - 当所需的序列生成完毕后，系统会发送 sleep 信号。
    - 这通常意味着释放推理引擎占用的资源，或者将 GPU 状态切换回 PPO Training 模式，以便利用刚才生成的数据进行梯度更新。

### hybrid

> wake_up 和 sleep 是 verl 在 Hybrid（混合）模式下实现高效资源利用的核心机制。

- two stages
    - Rollout 阶段：使用当前的策略模型生成数据（需要 vLLM/SGLang 这样的推理引擎）。
    - Train 阶段：使用生成的数据更新模型参数（需要 FSDP/Megatron 这样的训练引擎）。
- 资源
    - 训练引擎需要大量显存来存储梯度（Gradients）、优化器状态（Optimizer States）和激活值（Activations）。
    - 推理引擎需要大量显存来存储 KV Cache（上下文缓存），以加速生成。
- verl 采用了“分时复用”的策略：同一个 GPU，一会儿做训练，一会儿做推理。
- wake_up
    - 当 PPO 的训练步结束，准备开始生成新数据（Rollout）时，AgentLoopManager 会调用 wake_up。
        - 权重同步 (Weight Synchronization)：
            - 训练刚刚结束，模型的权重（Weights）已经被 FSDP/Megatron 更新了。
            - 推理引擎（vLLM/SGLang）里的权重还是旧的。
            - wake_up 指令会触发将最新的权重从训练引擎的内存空间复制/同步到推理引擎中。
        - 显存预留 (Memory Allocation / KV Cache Setup)：
            - 训练引擎暂停，释放部分临时显存。
            - 推理引擎“醒来”，申请显存用于 KV Cache，准备开始大规模的并发生成。
- sleep
    -  当所有的数据生成（Rollout）完成，准备进入 PPO 训练计算 Loss 和更新梯度时，AgentLoopManager 会调用 sleep。
        -  释放 KV Cache (Free KV Cache)：
            -  推理结束了，不再需要维护上下文缓存。
            -  sleep 指令强制 vLLM/SGLang 释放掉它占用的巨大 KV Cache 显存块。
        - 让路给训练 (Yield to Training)：
            - 显存被腾空，训练引擎（FSDP）接管 GPU，开始分配显存给梯度和优化器状态，进行反向传播。
         
```python
# verl/workers/rollout/vllm_rollout/vllm_async_server.py 
async def wake_up(self):
    if self.rollout_mode == RolloutMode.HYBRID:
        # 调用所有 worker 进行权重同步和显存准备
        await asyncio.gather(*[worker.wake_up.remote() for worker in self.workers])
    elif self.rollout_mode == RolloutMode.STANDALONE:
        # 如果是独立部署（推理和训练在不同GPU），则不需要做任何事
        logger.info("skip wake_up in standalone mode")

async def sleep(self):
    if self.rollout_mode == RolloutMode.HYBRID:
        # 重置 Prefix Cache，释放显存
        if self.node_rank == 0:
            await self.engine.reset_prefix_cache()
        # 让 worker 进入休眠状态
        await asyncio.gather(*[worker.sleep.remote() for worker in self.workers])
```

### test code 

- https://github.com/volcengine/verl/blob/main/tests/experimental/agent_loop/test_basic_agent_loop.py
```
pip install pytest-asyncio

# -q: quiet
# -s：允许测试代码中的 print 语句或日志直接输出到控制台。
pytest -q -s tests/experimental/agent_loop/test_basic_agent_loop.py
```